# 1. Checking Python version

In [1]:
!python -V

Python 3.12.11


## 2. Verify GPU & RAM Availability

In [2]:
import torch

if torch.cuda.is_available():
    print("GPU is available!")
    !nvidia-smi
    gpu_name = torch.cuda.get_device_name()
    print(f"GPU: '{gpu_name}'")
else:
    print("GPU is NOT available.")

GPU is not available.


In [6]:
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1024**3
print('Your runtime has {:.1f}GB of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('NOT using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 12.7GB of available RAM

Not using a high-RAM runtime


# 3. Upload Image Dataset and Prepare Training Data

## 3.1 Upload images

**Option 1 - Copy from Google Drive**



In [7]:
# path of data.zip in Google Drive
data_zip_path = '/content/drive/MyDrive/git_personal/yolo-custom-object-detection-training/data.zip'

In [3]:
from google.colab import drive
drive.mount('/content/drive')

!cp {data_zip_path} /content

Mounted at /content/drive


## 3.2 Split images into train and validation folders

In [4]:
# Unzip images to a custom data folder
!unzip -q /content/data.zip -d /content/custom_data

In [5]:
!wget -O /content/train_val_split.py https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py

--2025-08-20 14:39:15--  https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3203 (3.1K) [text/plain]
Saving to: ‘/content/train_val_split.py’

/content/train_val_ 100%[===================>]   3.13K  --.-KB/s    in 0s      

2025-08-20 14:39:15 (50.4 MB/s) - ‘/content/train_val_split.py’ saved [3203/3203]



In [6]:
# TO DO: Improve robustness of train_val_split.py script so it can handle nested data folders, etc
!python train_val_split.py --datapath="/content/custom_data" --train_pct=0.9

Created folder at /content/data/train/images.
Created folder at /content/data/train/labels.
Created folder at /content/data/validation/images.
Created folder at /content/data/validation/labels.
Number of image files: 28
Number of annotation files: 28
Images moving to train: 25
Images moving to validation: 3


# 4. Install Requirements (Ultralytics)

In [7]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.9 MB/s eta 0:00:00


## Configure Training

In [8]:
# Python function to automatically create data.yaml config file
# 1. Reads "classes.txt" file to get list of class names
# 2. Creates data dictionary with correct paths to folders, number of classes, and names of classes
# 3. Writes data in YAML format to data.yaml

import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  # Read class.txt to get class names
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Create data dictionary
  data = {
      'path': '/content/data',
      'train': 'train/images',
      'val': 'validation/images',
      'nc': number_of_classes,
      'names': classes
  }

  # Write data to YAML file
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Define path to classes.txt and run function
path_to_classes_txt = '/content/custom_data/classes.txt'
path_to_data_yaml = '/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
!cat /content/data.yaml

Created config file at /content/data.yaml

File contents:

path: /content/data
train: train/images
val: validation/images
nc: 1
names:
- dog


In [33]:
# clean up labels cache (if any)
label_cache_paths = ['/content/data/train/labels.cache', '/content/data/validation/labels.cache']

for label_cache_path in label_cache_paths:
  if os.path.isfile(label_cache_path):
    os.remove(label_cache_path)

# Train Model

In [37]:
def elapsed_time(elapsed_sec):
  hours = int(elapsed_sec // 3600)
  minutes = int((elapsed_sec % 3600) // 60)
  seconds = int(elapsed_sec % 60)

  # Display the execution time
  print(f"Execution time: {hours}hr {minutes}min {seconds}sec")

In [47]:
gpu_name = torch.cuda.get_device_name()
print(f"Using GPU '{gpu_name}'")

# batch: 16, 20, 32, 40, 64, 80, 96, 128
# worker: 8

if 'A100' in gpu_name:
  batch = 128
  workers = 24
else:
  batch = 128
  workers = 24

model = 'yolo11s.pt'
batch = 128
workers = 24
epochs = 400
imgsz = 640
save_period = -1 # default -1
patience = 200 # default 100, 0 to disable
cache = True # default False, cache dataset images in memory

print(f"batch: {batch}, workers: {workers}, epochs: {epochs}")

RuntimeError: No CUDA GPUs are available

In [38]:
import time
from datetime import datetime
import pytz

def get_datetime(timezone='Etc/GMT-8'):
    tz = pytz.timezone(timezone)
    return datetime.now(tz)

# measure execution time
start_time = time.time()
print(f"Start training at {get_datetime()}")

Start training at 2025-08-20 23:28:07.240698+08:00


In [35]:
!yolo detect train data={path_to_data_yaml} model={model} epochs={epochs} imgsz={imgsz} cache={cache} workers={workers} batch={batch} patience={patience} save_period={save_period} exist_ok=True

Traceback (most recent call last):
  File "/usr/local/bin/yolo", line 5, in <module>
    from ultralytics.cfg import entrypoint
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/__init__.py", line 11, in <module>
    from ultralytics.models import NAS, RTDETR, SAM, YOLO, YOLOE, FastSAM, YOLOWorld
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/models/__init__.py", line 3, in <module>
    from .fastsam import FastSAM
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/models/fastsam/__init__.py", line 3, in <module>
    from .model import FastSAM
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/models/fastsam/model.py", line 6, in <module>
    from ultralytics.engine.model import Model
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 8, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 2193, in <module>
  File "/usr/local/lib/python3.12/dist-packages/torch/neste

In [ ]:
elapsed_time(time.time() - start_time)

# Test Model

In [25]:
!yolo detect predict model=runs/detect/train/weights/best.pt source=data/validation/images save=True

Traceback (most recent call last):
  File "/usr/local/bin/yolo", line 8, in <module>
    sys.exit(entrypoint())
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/__init__.py", line 956, in entrypoint
    model = YOLO(model, task=task)
            ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/models/yolo/model.py", line 81, in __init__
    super().__init__(model=model, task=task, verbose=verbose)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 151, in __init__
    self._load(model, task=task)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 295, in _load
    self.model, self.ckpt = attempt_load_one_weight(weights)
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/tasks.py", line 1549, in attempt_load_one_weight
    ckpt, weight = torch_safe_load(weight)  # load ckpt
    

In [26]:
import glob
from IPython.display import Image, display
for image_path in glob.glob(f'/content/runs/detect/predict/*.jpg')[:10]:
  display(Image(filename=image_path, height=400))
  print('\n')

# Download YOLO Model

In [27]:
# Create "my_model" folder to store model weights and train results
!mkdir /content/my_model
!cp /content/runs/detect/train/weights/best.pt /content/my_model/my_model.pt
!cp -r /content/runs/detect/train /content/my_model

cp: cannot stat '/content/runs/detect/train/weights/best.pt': No such file or directory


In [28]:
# Export Model to NCNN Format
from ultralytics import YOLO

# Load your trained YOLOv11 model
model = YOLO("/content/my_model/my_model.pt")

# Export the model to NCNN format
model.export(format="ncnn")

FileNotFoundError: [Errno 2] No such file or directory: '/content/my_model/my_model.pt'

In [ ]:
# Zip into "my_model.zip"
%cd my_model
!zip /content/my_model.zip my_model.pt
!zip -r /content/my_model.zip my_model_ncnn_model
!zip -r /content/my_model.zip train
%cd /content

In [ ]:
# This takes forever for some reason, you can also just download the model from the sidebar
from google.colab import files

files.download('/content/my_model.zip')